# Rogue Handoff Simulation — a toy model of the RogueHandoff-20 finding

Tencent Zhuque Lab's **RogueHandoff-20** benchmark (contributed via GitHub PR to
Tencent's AI-Infra-Guard) reports a striking result: on normal tasks agents show
**0–5%** harm rates, but after one unsafe agent-to-agent handoff injection the
receiving agent executes harmful actions **40–95%** of the time — even though
the request it actually sees looks completely clean.

This notebook is a **seeded toy simulation** of that mechanism, not a
replication of the benchmark:

1. Agents that refuse overtly malicious prompts (~97%) still fail when harmful
   intent arrives *through the handoff*.
2. Compromise compounds with chain length: `P = 1 - (1 - p)^k`.
3. A provenance checkpoint at each handoff drops the effective per-hop rate
   and flattens the curve.


In [ ]:
import numpy as np

rng = np.random.default_rng(42)
N = 200_000

# Direct attack: agent sees an overtly malicious request and can refuse it.
# Benchmark baseline harm on normal tasks is 0-5%; agents reliably refuse
# overtly harmful prompts, so model refusal success at 97%.
refusal_ok = rng.random(N) < 0.97
direct_harm_rate = 1 - refusal_ok.mean()

# Handoff attack: router injects an unsafe trajectory; the receiver's own input
# looks clean, so refusal never fires. Model the four tested routes spanning
# the reported 40-95% band.
route_rates = np.array([0.40, 0.62, 0.78, 0.95])
handoff_harm = rng.random((4, N)) < route_rates[:, None]
handoff_harm_rates = handoff_harm.mean(axis=1)

print(f"Direct-injection harm rate : {direct_harm_rate*100:.2f}%  (refusal works)")
for r, hr in zip(route_rates, handoff_harm_rates):
    print(f"Handoff route (true p={r:.2f}) harm : {hr*100:.2f}%")

assert abs(direct_harm_rate - 0.03) < 0.005, "refusal model off"
assert (handoff_harm_rates >= 0.39).all() and (handoff_harm_rates <= 0.96).all(), \
    "handoff rates should sit in the reported 40-95% band"
print("PASS: refusal blocks direct attacks; handoffs bypass it in the 40-95% band")


## Compromise compounds with chain length

If each hop infects with probability `p`, a `k`-hop pipeline is compromised with
probability `1 - (1 - p)^k`. The worst tested route (p=0.95) and the best
(p=0.40) are both measured against a defended route with a provenance
checkpoint (p=0.05).


In [ ]:
hops = np.arange(1, 7)
cases = {"worst route (p=0.95)": 0.95,
         "best route (p=0.40)": 0.40,
         "provenance checkpoint (p=0.05)": 0.05}
table = {name: 1 - (1 - p) ** hops for name, p in cases.items()}

print(f"{'hops':>4}  " + "  ".join(f"{n:>28}" for n in table))
for k in hops:
    print(f"{k:>4}  " + "  ".join(f"{v[k-1]*100:>27.2f}%" for v in table.values()))

worst, best, defended = table["worst route (p=0.95)"], table["best route (p=0.40)"], \
    table["provenance checkpoint (p=0.05)"]
assert abs(worst[2] - 0.9999) < 1e-4, "hop-3 worst-route value"
assert abs(best[2] - 0.7840) < 1e-4, "hop-3 best-route value"
assert defended[5] < 0.27, "hop-6 defended chain must stay under 27%"
print("PASS: chain-length compounding verified (worst 99.99% at 3 hops; defended 26.49% at 6 hops)")


## The defense: verify the handoff, not just the request

Simulate a provenance checkpoint that re-screens handoff context with
sensitivity 0.95 — catching 95% of injected trajectories before the receiver
acts. Effective per-hop rate drops to `p * (1 - 0.95)`.


In [ ]:
p_worst, sensitivity = 0.95, 0.95
p_effective = p_worst * (1 - sensitivity)   # 0.0475
k = 6
comp_undefended = 1 - (1 - p_worst) ** k
comp_defended = 1 - (1 - p_effective) ** k
print(f"6-hop chain, worst route : {comp_undefended*100:.2f}% compromised")
print(f"6-hop chain, with 95%-sensitive provenance checkpoint: {comp_defended*100:.2f}% compromised")
print(f"Risk reduction factor: {comp_undefended/comp_defended:.1f}x")
assert comp_undefended > 0.999 and comp_defended < 0.30
print("PASS: handoff-level verification beats per-request refusal for multi-agent chains")
